[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C48_Cloud_Deployment_Course/02_serving_api/02_serving_api.ipynb)

# 02 · 推理服务 API（契约校验、SSE 流式、限流与容量模型，全部从零实现）

目标：把 **OpenAI 兼容契约 → SSE 编解码 → 令牌桶限流 → M/M/c 容量模型** 从零写出来，
每个组件都**对拍**朴素参考、每个结论都**算一笔账**。

路线：请求校验器 → SSE round-trip → TTFT/TPOT 拆解 → 令牌桶 vs 固定窗口 → Erlang-C → 副本数规划 → ✏️ 练习 → 📖 答案 → 🧪 真实 SLO 胶囊。

> 心智模型：**契约定义能力边界，护栏定义失效行为，队列论定义容量**。
> 本环境不起 HTTP 服务，但这三件事的全部逻辑都是纯计算，可以精确复现。

## 1 · 契约：OpenAI 兼容请求的校验器

契约的第一职责是**在最便宜的地方拒绝坏请求**。写一个校验器，
并验证它对合法请求放行、对各类非法请求给出**精确**的错误（不是笼统的 400）。

In [ ]:
import math, json, random, heapq
from dataclasses import dataclass, field

VALID_ROLES = {'system', 'user', 'assistant', 'tool'}

def validate_chat_request(req, max_context=8192, max_output_cap=4096):
    '''返回 (ok, error_dict_or_None)。仿 OpenAI 的错误结构。'''
    def err(msg, param, code):
        return False, {'error': {'message': msg, 'param': param, 'type': 'invalid_request_error', 'code': code}}

    if 'model' not in req:
        return err('missing required field', 'model', 'missing_field')
    msgs = req.get('messages')
    if not isinstance(msgs, list) or not msgs:
        return err('messages must be a non-empty list', 'messages', 'invalid_type')
    for i, m in enumerate(msgs):
        if m.get('role') not in VALID_ROLES:
            return err(f'invalid role at index {i}', 'messages', 'invalid_role')
        if not isinstance(m.get('content', None), str):
            return err(f'content must be string at index {i}', 'messages', 'invalid_type')
    t = req.get('temperature', 1.0)
    if not (0.0 <= t <= 2.0):
        return err('temperature must be in [0, 2]', 'temperature', 'out_of_range')
    n_out = req.get('max_tokens', 512)
    if n_out > max_output_cap:
        return err(f'max_tokens exceeds cap {max_output_cap}', 'max_tokens', 'out_of_range')
    # 上下文预算：粗估 prompt token = 字符数/4（真实用 tokenizer，见 C50）
    est_prompt = sum(len(m['content']) for m in msgs) // 4
    if est_prompt + n_out > max_context:
        return err(f'context overflow: {est_prompt}+{n_out} > {max_context}', 'messages', 'context_length_exceeded')
    return True, None

good = {'model': 'llama-3-8b', 'messages': [{'role': 'user', 'content': '你好'}], 'max_tokens': 128}
ok, e = validate_chat_request(good); print('合法请求:', ok)
assert ok and e is None

cases = [
    ({'messages': [{'role':'user','content':'x'}]},                          'missing_field'),
    ({'model':'m', 'messages': []},                                          'invalid_type'),
    ({'model':'m', 'messages': [{'role':'wizard','content':'x'}]},           'invalid_role'),
    ({'model':'m', 'messages': [{'role':'user','content':'x'}], 'temperature': 3.0}, 'out_of_range'),
    ({'model':'m', 'messages': [{'role':'user','content':'x'*40000}]},       'context_length_exceeded'),
]
for req, expected_code in cases:
    ok, e = validate_chat_request(req)
    assert not ok and e['error']['code'] == expected_code, (req, e)
    print(f"  ✓ {expected_code:<26s} -> {e['error']['message']}")
print('✅ 校验器给出的是「哪个字段、为什么」，不是笼统的 400')

### 健康端点的语义分离

**liveness 失败 = 重启我；readiness 失败 = 别给我流量（但别重启我）。**

判据：**重启能修复 → liveness；等待或减流能修复 → readiness**。
下面把这条判据编码成状态机，并验证「模型加载中」必须是 `alive=True, ready=False`。

In [ ]:
@dataclass
class ServerState:
    weights_loaded: bool = False
    event_loop_alive: bool = True
    queue_depth: int = 0
    max_queue: int = 64
    shutting_down: bool = False
    downstream_ok: bool = True      # 下游依赖（如 tokenizer 服务）

def healthz(s):     # liveness：只看进程内不可自愈的状态
    return 200 if s.event_loop_alive else 500

def readyz(s):      # readiness：看「现在能不能接流量」
    if s.shutting_down:            return 503   # 优雅停机第一步
    if not s.weights_loaded:       return 503   # 加载中：活着但没准备好
    if s.queue_depth >= s.max_queue: return 503 # 过载：暂时摘流量
    return 200

loading = ServerState(weights_loaded=False)
assert healthz(loading) == 200, '模型加载中，进程是活的 —— 绝不能让 liveness 失败'
assert readyz(loading) == 503,  '模型加载中，不能接流量'
print('加载中: liveness 200 (别重启我) / readiness 503 (别给我流量) ✅')

ready = ServerState(weights_loaded=True)
assert (healthz(ready), readyz(ready)) == (200, 200)

overload = ServerState(weights_loaded=True, queue_depth=64)
assert healthz(overload) == 200 and readyz(overload) == 503, '过载应摘流量而非重启'

dead = ServerState(weights_loaded=True, event_loop_alive=False)
assert healthz(dead) == 500, '事件循环卡死只能靠重启'

# 反面教材：把下游依赖检查写进 liveness
def bad_healthz(s): return 200 if (s.event_loop_alive and s.downstream_ok) else 500
flaky = ServerState(weights_loaded=True, downstream_ok=False)
assert bad_healthz(flaky) == 500 and healthz(flaky) == 200
print('⚠️  反面教材：下游抖动 -> bad_healthz 返回 500 -> 全部 Pod 被重启 -> 故障放大')
print('✅ 语义分离正确：liveness 只测自己，readiness 测「现在能否服务」')

## 2 · SSE：编码、解码与 round-trip 对拍

SSE 规则极简：每条事件 `data: <payload>\n\n`（**空行**结束），OpenAI 用 `data: [DONE]` 收尾。
写出编码器和解码器，用 **round-trip 对拍**验证协议实现正确。

In [ ]:
def sse_encode(chunks, done_sentinel='[DONE]'):
    '''把一串 delta 对象编码成 SSE 字节流。'''
    out = []
    for c in chunks:
        out.append('data: ' + json.dumps(c, ensure_ascii=False) + '\n\n')
    out.append(f'data: {done_sentinel}\n\n')
    return ''.join(out)

def sse_decode(stream, done_sentinel='[DONE]'):
    '''解析 SSE 流，返回 delta 对象列表（遇到 [DONE] 停止）。'''
    events = []
    for block in stream.split('\n\n'):
        line = block.strip()
        if not line or not line.startswith('data: '):
            continue
        payload = line[len('data: '):]
        if payload == done_sentinel:
            break
        events.append(json.loads(payload))
    return events

def make_deltas(text_tokens):
    ds = [{'choices': [{'index': 0, 'delta': {'role': 'assistant'}}]}]
    ds += [{'choices': [{'index': 0, 'delta': {'content': t}}]} for t in text_tokens]
    ds += [{'choices': [{'index': 0, 'delta': {}, 'finish_reason': 'stop'}]}]
    return ds

toks = ['LSH', ' 是', '一种', '局部', '敏感', '哈希']
deltas = make_deltas(toks)
wire = sse_encode(deltas)
print(wire[:160].replace('\n', '⏎') + ' …')

# round-trip 对拍：解码回来必须与原始 deltas 完全一致
assert sse_decode(wire) == deltas, 'SSE round-trip 必须无损'
# 从 delta 流重建完整文本（客户端要做的事）
text = ''.join(d['choices'][0]['delta'].get('content', '') for d in sse_decode(wire))
assert text == ''.join(toks), text
print(f'\n重建文本: {text!r}')
print('✅ SSE round-trip 对拍通过')

### 中间层缓冲：为什么本地 curl 正常、上线后不流式

代理若开启缓冲，会攒够 N 字节才转发一次。**总时长不变，但 TTFT 被毁掉。**
下面量化这个差别。

In [ ]:
def simulate_delivery(wire, ttft_ms=300, tpot_ms=40, n_tokens=6, buffer_bytes=0):
    '''返回每个 token 到达客户端的时刻(ms)。buffer_bytes=0 表示不缓冲。'''
    per_event = len(wire) / (n_tokens + 2)     # 粗略：每条事件的字节数
    arrivals, pending, t = [], 0.0, ttft_ms
    for i in range(n_tokens):
        pending += per_event
        if buffer_bytes == 0 or pending >= buffer_bytes:
            arrivals.append(t); pending = 0.0
        else:
            arrivals.append(None)              # 还被缓冲着，没送出去
        t += tpot_ms
    # 缓冲未满的部分在流结束时一次性 flush
    flush_at = ttft_ms + tpot_ms * n_tokens
    return [a if a is not None else flush_at for a in arrivals]

no_buf = simulate_delivery(wire, buffer_bytes=0)
buffered = simulate_delivery(wire, buffer_bytes=4096)    # nginx 默认缓冲区量级
print('无缓冲, 各 token 到达(ms):', [round(x) for x in no_buf])
print('有缓冲, 各 token 到达(ms):', [round(x) for x in buffered])
print(f'TTFT: 无缓冲 {no_buf[0]:.0f}ms  vs  有缓冲 {buffered[0]:.0f}ms')
assert buffered[0] > no_buf[0] * 1.5, '缓冲会显著恶化 TTFT'
assert all(b >= a for a, b in zip(no_buf, buffered)), '缓冲只会让每个 token 更晚到达'
assert len(set(buffered)) == 1, '被缓冲时所有 token 在流结束时一次性抵达——流式名存实亡'
print('✅ 生成总时长没变，被毁掉的只有「边生成边看到」这个体验。')
print('   这就是必须设 proxy_buffering off / X-Accel-Buffering: no 的原因')

## 3 · TTFT / TPOT：为什么延迟指标必须拆开

$$L_{e2e} = W_{queue} + T_{prefill}(n_{prompt}) + n_{out} \times T_{decode}$$

只看端到端延迟会被**输出长度分布**完全主导——一个爱写长文的用户就能把你的 P95 搞崩。

In [ ]:
def e2e_latency(w_queue_ms, n_prompt, n_out, prefill_ms_per_1k=90.0, tpot_ms=35.0):
    t_prefill = n_prompt / 1000 * prefill_ms_per_1k
    return w_queue_ms + t_prefill + n_out * tpot_ms

rng = random.Random(0)
# 两类用户：短问答 vs 长文生成，服务端能力完全相同
short = [e2e_latency(20, 300, rng.randint(30, 120)) for _ in range(2000)]
long_ = [e2e_latency(20, 300, rng.randint(800, 2000)) for _ in range(2000)]

def p(xs, q):
    s = sorted(xs); return s[min(len(s)-1, int(q * len(s)))]

print(f'短问答  P50 {p(short,.5)/1000:>6.2f}s  P95 {p(short,.95)/1000:>6.2f}s')
print(f'长文    P50 {p(long_,.5)/1000:>6.2f}s  P95 {p(long_,.95)/1000:>6.2f}s')
mixed = short + long_
print(f'混合    P50 {p(mixed,.5)/1000:>6.2f}s  P95 {p(mixed,.95)/1000:>6.2f}s  ← 被长文主导')

assert p(long_, .95) > 5 * p(short, .95), '同样的服务端能力，P95 差 5 倍以上'
# 而 TTFT 和 TPOT 对两类用户是一样的（服务端能力的真实反映）
ttft_short = 20 + 300/1000*90
ttft_long  = 20 + 300/1000*90
assert abs(ttft_short - ttft_long) < 1e-9
print('\n✅ TTFT/TPOT 对两类用户完全相同 —— 它们才是服务端能力的指标。')
print('   SLO 必须写成「P95 TTFT < 800ms 且 P95 TPOT < 60ms」，而非端到端。')

## 4 · 限流：令牌桶 vs 固定窗口

**固定窗口计数器**有个致命缺陷：窗口边界可以放过 **2 倍**的突发。
令牌桶没有这个问题。下面把两者都实现，用同一串请求对比。

In [ ]:
class TokenBucket:
    def __init__(self, rate, capacity):
        self.rate, self.cap = rate, capacity
        self.tokens, self.last = float(capacity), 0.0
    def allow(self, now, cost=1.0):
        self.tokens = min(self.cap, self.tokens + (now - self.last) * self.rate)
        self.last = now
        if self.tokens >= cost:
            self.tokens -= cost; return True
        return False

class FixedWindow:
    def __init__(self, limit, window=1.0):
        self.limit, self.window, self.count, self.win_start = limit, window, 0, 0.0
    def allow(self, now, cost=1.0):
        if now - self.win_start >= self.window:
            self.win_start, self.count = now - (now % self.window), 0
        if self.count + cost <= self.limit:
            self.count += cost; return True
        return False

# 攻击场景：在窗口边界前后各打满 10 个请求
RATE = 10
tb, fw = TokenBucket(RATE, RATE), FixedWindow(RATE, 1.0)
burst = [0.95]*10 + [1.00]*10        # 边界前 10 个 + 边界后 10 个
tb_pass = sum(tb.allow(t) for t in burst)
fw_pass = sum(fw.allow(t) for t in burst)
print(f'边界突发 20 个请求 (限速 {RATE}/s): 令牌桶放行 {tb_pass}, 固定窗口放行 {fw_pass}')
assert fw_pass > tb_pass, '固定窗口在边界会放过约 2 倍流量'
print('✅ 固定窗口的边界效应：0.95s~1.00s 这 50ms 内放过了 2 倍额度')

# 对拍：长期速率必须收敛到 rate
tb2 = TokenBucket(RATE, RATE)
passed = sum(tb2.allow(i * 0.01) for i in range(1000))    # 10 秒内每 10ms 打一个
expected = RATE * 10 + RATE                                # 长期 rate*T + 初始桶容量
assert abs(passed - expected) <= 2, f'长期放行应≈{expected}，得到 {passed}'
print(f'✅ 令牌桶长期速率收敛: 10 秒放行 {passed} ≈ rate*10 + 桶容量 = {expected}')

### 为什么 LLM 必须按 token 限流

一个请求可能生成 10 个 token，也可能 4000 个——资源消耗差 400 倍。
**只限 RPM 的服务一定会被长生成打爆。**

In [ ]:
def simulate_rpm_only(requests, rpm_limit):
    '''只限请求数：返回实际消耗的 token 总量。桶容量 = 一分钟额度。'''
    tb = TokenBucket(rpm_limit / 60.0, rpm_limit)
    return sum(n_tok for t, n_tok in requests if tb.allow(t))

def simulate_rpm_tpm(requests, rpm_limit, tpm_limit):
    '''RPM + TPM 双限。'''
    tb_r = TokenBucket(rpm_limit / 60.0, rpm_limit)
    tb_t = TokenBucket(tpm_limit / 60.0, tpm_limit)
    used = 0
    for t, n_tok in requests:
        if tb_r.allow(t) and tb_t.allow(t, cost=n_tok):
            used += n_tok
    return used

rng = random.Random(1)
normal = [(i * 0.1, rng.randint(50, 200)) for i in range(600)]        # 正常用户
abuser = [(i * 0.1, 4000) for i in range(600)]                        # 每个请求都顶格生成

RPM, TPM = 60, 60_000
print(f'限额: RPM={RPM}, TPM={TPM:,}')
for name, reqs in [('正常用户', normal), ('长生成滥用', abuser)]:
    only_rpm = simulate_rpm_only(reqs, RPM)
    both     = simulate_rpm_tpm(reqs, RPM, TPM)
    print(f'  {name:<12s} 仅限RPM放过 {only_rpm:>8,} tok | RPM+TPM放过 {both:>8,} tok')

abuse_rpm_only = simulate_rpm_only(abuser, RPM)
abuse_both     = simulate_rpm_tpm(abuser, RPM, TPM)
assert abuse_rpm_only > 3 * abuse_both, '仅限 RPM 时，长生成能绕过限额数倍'
print(f'\n✅ 滥用者在「仅限 RPM」下多消耗 {abuse_rpm_only/abuse_both:.1f} 倍算力 —— 必须双限')

## 5 · 容量模型：Erlang-C 与副本数规划

M/M/c 的核心结论：**排队时间是利用率的双曲函数，ρ→1 时爆炸**。
先实现 Erlang-C，用蒙特卡洛离散事件仿真**对拍**它，再拿它做容量规划。

In [ ]:
def erlang_c(c, a):
    '''到达时需要排队的概率。a = λ/μ（提供负载，单位 erlang）。'''
    if a >= c:
        return 1.0
    s = sum(a**k / math.factorial(k) for k in range(c))
    top = a**c / math.factorial(c) * (c / (c - a))
    return top / (s + top)

def mmc_wait(lam, mu, c):
    '''M/M/c 平均排队等待时间 W_q。'''
    a = lam / mu
    if a >= c:
        return float('inf')
    return erlang_c(c, a) / (c * mu - lam)

MU = 2.0                      # 单副本每秒处理 2 个请求
print(f"{'c':>3s} {'λ':>6s} {'ρ':>6s} {'W_q(ms)':>10s}")
for c in [1, 4, 10]:
    lam = 0.8 * c * MU        # 固定 ρ=0.8
    print(f'{c:>3d} {lam:>6.1f} {lam/(c*MU):>6.2f} {mmc_wait(lam,MU,c)*1000:>10.1f}')

w1, w10 = mmc_wait(0.8*1*MU, MU, 1), mmc_wait(0.8*10*MU, MU, 10)
assert w10 < w1 / 3, '同样 ρ=0.8，c=10 的排队应远小于 c=1（规模经济）'
print(f'\n✅ 规模经济：同样 ρ=0.8，c=1 排队 {w1*1000:.0f}ms，c=10 只有 {w10*1000:.0f}ms')
print('   这就是「一个 10 副本共享池」优于「10 个独立单副本服务」的数学原因。')

In [ ]:
# 对拍：离散事件仿真 vs Erlang-C 解析解
def simulate_mmc(lam, mu, c, n_req=60000, seed=0):
    rng = random.Random(seed)
    free_at = [0.0] * c                     # 每个服务位的空闲时刻
    t, waits = 0.0, []
    for _ in range(n_req):
        t += rng.expovariate(lam)           # 泊松到达
        heapq.heapify(free_at)
        earliest = free_at[0]
        w = max(0.0, earliest - t)
        waits.append(w)
        heapq.heapreplace(free_at, max(t, earliest) + rng.expovariate(mu))
    return sum(waits) / len(waits)

for c, rho in [(1, 0.7), (4, 0.8), (8, 0.85)]:
    lam = rho * c * MU
    sim, ana = simulate_mmc(lam, MU, c), mmc_wait(lam, MU, c)
    rel = abs(sim - ana) / ana
    print(f'c={c} ρ={rho}: 仿真 {sim*1000:>7.1f}ms | 解析 {ana*1000:>7.1f}ms | 相对误差 {rel:.1%}')
    assert rel < 0.15, f'仿真与解析应吻合，c={c} 误差 {rel:.1%}'
print('✅ 对拍通过：Erlang-C 解析解与离散事件仿真一致（<15%）')

### 死亡地带：ρ 从 0.8 推到 0.95 会发生什么

In [ ]:
C = 8
print(f"{'ρ':>6s} {'λ(QPS)':>8s} {'W_q(ms)':>10s} {'相对 ρ=0.7':>12s}")
base = mmc_wait(0.7*C*MU, MU, C)
for rho in [0.7, 0.8, 0.85, 0.9, 0.95, 0.98]:
    w = mmc_wait(rho*C*MU, MU, C)
    print(f'{rho:>6.2f} {rho*C*MU:>8.1f} {w*1000:>10.1f} {w/base:>11.1f}×')

w70, w95 = mmc_wait(0.7*C*MU, MU, C), mmc_wait(0.95*C*MU, MU, C)
assert w95 > 8 * w70, 'ρ 0.7→0.95 排队应恶化近一个数量级'
print(f'\n✅ 多榨 36% 吞吐，排队时间涨 {w95/w70:.0f} 倍。')
print('   生产目标利用率 0.6~0.8 不是保守，是「延迟稳定性」的合理定价。')

## ✏️ 练习 1：满足 TTFT SLO 的最小副本数

实现 `min_replicas(lam, mu, ttft_budget_s, prefill_s)`：
返回使 **W_q + prefill ≤ ttft_budget_s** 的**最小副本数 c**（从 1 开始试，`c` 上限 1000）。
若无解返回 `None`。

In [ ]:
def min_replicas(lam, mu, ttft_budget_s, prefill_s):
    # TODO: 从 c=1 试到 1000，找第一个满足 mmc_wait(lam, mu, c) + prefill_s <= ttft_budget_s 的 c
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
c = min_replicas(lam=12.0, mu=2.0, ttft_budget_s=0.8, prefill_s=0.3)
assert c is not None and c > 6, f'λ/μ=6，至少要 7 个副本才不排到无穷，得到 {c}'
assert mmc_wait(12.0, 2.0, c) + 0.3 <= 0.8 + 1e-9, 'c 必须真的满足预算'
assert c == 1 or mmc_wait(12.0, 2.0, c-1) + 0.3 > 0.8, 'c 必须是最小的那个'
# 预算越紧，需要越多副本
c_tight = min_replicas(12.0, 2.0, 0.35, 0.3)
c_loose = min_replicas(12.0, 2.0, 3.0, 0.3)
assert c_tight > c_loose, '更紧的 TTFT 预算需要更多副本'
print(f'λ=12 μ=2: TTFT预算 0.8s -> {c} 副本 | 0.35s -> {c_tight} 副本 | 3.0s -> {c_loose} 副本')
print('✅ 练习 1 通过：SLO 直接翻译成副本数')

## ✏️ 练习 2：按 token 计费的令牌桶

实现 `TokenBudget`：一个按 **token** 而非请求计数的限流器。
接口：`__init__(self, tpm)`（每分钟 token 额度，桶容量 = tpm）、
`try_reserve(self, now, est_tokens)` 预扣、`settle(self, actual_tokens, est_tokens)` 结算（退还多扣的）。

In [ ]:
class TokenBudget:
    def __init__(self, tpm):
        # TODO: rate = tpm/60 tok/s, capacity = tpm；self.tokens 初始为满
        raise NotImplementedError
    def try_reserve(self, now, est_tokens):
        # TODO: 补充令牌(受 capacity 上限)、够则扣除返回 True，否则 False
        raise NotImplementedError
    def settle(self, actual_tokens, est_tokens):
        # TODO: 退还 (est - actual)，但总量不超过 capacity
        raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
tb = TokenBudget(tpm=6000)              # 100 tok/s
assert tb.try_reserve(0.0, 1000) is True
assert tb.try_reserve(0.0, 5000) is True     # 桶容量 6000，刚好用完
assert tb.try_reserve(0.0, 1)     is False,  '额度用尽应拒绝'
tb.settle(actual_tokens=200, est_tokens=1000)   # 实际只用了 200，退还 800
assert tb.try_reserve(0.0, 800) is True, '退还后应能再预扣 800'
# 一分钟后应恢复满额
tb2 = TokenBudget(tpm=6000)
tb2.try_reserve(0.0, 6000)
assert tb2.try_reserve(60.0, 6000) is True, '60 秒后应补满 6000'
print('✅ 练习 2 通过：预扣-结算模型让「先限流后知道实际用量」成为可能')

## ✏️ 练习 3：优雅降级的准入决策

实现 `admit(queue_depth, max_queue, tier)`：三档租户 `tier ∈ {'premium','standard','free'}`。
规则：队列 < 50% 全部放行；50%~80% 拒 `free`；80%~100% 只放 `premium`；≥100% 全拒。
返回 `(bool, status_code)`，放行 `(True, 200)`，拒绝 `(False, 429)`。

In [ ]:
def admit(queue_depth, max_queue, tier):
    # TODO: 按上述四档规则返回 (allowed, status)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
MAXQ = 100
assert admit(10,  MAXQ, 'free')     == (True, 200)
assert admit(60,  MAXQ, 'free')     == (False, 429)
assert admit(60,  MAXQ, 'standard') == (True, 200)
assert admit(90,  MAXQ, 'standard') == (False, 429)
assert admit(90,  MAXQ, 'premium')  == (True, 200)
assert admit(100, MAXQ, 'premium')  == (False, 429)
# 单调性：队列越深，放行的档次只减不增
for tier in ['premium', 'standard', 'free']:
    allowed = [admit(q, MAXQ, tier)[0] for q in range(0, 101, 10)]
    assert allowed == sorted(allowed, reverse=True), f'{tier} 的放行应随队列深度单调不增'
print('✅ 练习 3 通过：过载时先牺牲低优先级，而不是全体一起慢')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def min_replicas(lam, mu, ttft_budget_s, prefill_s):
    for c in range(1, 1001):
        if mmc_wait(lam, mu, c) + prefill_s <= ttft_budget_s:
            return c
    return None

In [ ]:
# 练习 2 参考答案
class TokenBudget:
    def __init__(self, tpm):
        self.rate = tpm / 60.0
        self.cap = float(tpm)
        self.tokens = float(tpm)
        self.last = 0.0
    def try_reserve(self, now, est_tokens):
        self.tokens = min(self.cap, self.tokens + (now - self.last) * self.rate)
        self.last = now
        if self.tokens >= est_tokens:
            self.tokens -= est_tokens
            return True
        return False
    def settle(self, actual_tokens, est_tokens):
        self.tokens = min(self.cap, self.tokens + max(0, est_tokens - actual_tokens))

In [ ]:
# 练习 3 参考答案
def admit(queue_depth, max_queue, tier):
    r = queue_depth / max_queue
    if r >= 1.0:                              return (False, 429)
    if r >= 0.8:  return (True, 200) if tier == 'premium' else (False, 429)
    if r >= 0.5:  return (False, 429) if tier == 'free' else (True, 200)
    return (True, 200)

---
## 🧪 真实数据胶囊：把 SLO 翻译成一张容量表

给定真实量级的服务参数，产出一张「QPS → 副本数 → 月成本 → P95 TTFT」的规划表。
这张表就是你去和产品/财务谈判时手里的东西。
（本环境不联网，单价用公开量级，可改成你自己的报价。）

In [ ]:
# 公开量级：7B 模型在单张 H100 上的典型服务能力
MU_PER_REPLICA = 2.5          # req/s（含 prefill+decode，平均输出 300 token）
PREFILL_S      = 0.28         # 平均 prompt 1.5k token 的 prefill
GPU_HOURLY     = 4.0          # 单卡 H100 按需 $/h（量级）
TTFT_SLO_S     = 0.8

print(f"{'峰值QPS':>8s} {'副本':>5s} {'ρ':>6s} {'W_q(ms)':>9s} {'TTFT(ms)':>10s} {'月成本$':>10s}")
plan = []
for qps in [5, 10, 25, 50, 100, 200]:
    c = min_replicas(qps, MU_PER_REPLICA, TTFT_SLO_S, PREFILL_S)
    wq = mmc_wait(qps, MU_PER_REPLICA, c)
    ttft = (wq + PREFILL_S) * 1000
    cost = c * GPU_HOURLY * 24 * 30
    plan.append((qps, c, qps/(c*MU_PER_REPLICA), ttft, cost))
    print(f'{qps:>8d} {c:>5d} {qps/(c*MU_PER_REPLICA):>6.2f} {wq*1000:>9.1f} {ttft:>10.1f} {cost:>10,.0f}')

# 性质检查
qpss  = [r[0] for r in plan]; reps = [r[1] for r in plan]; costs = [r[4] for r in plan]
assert reps == sorted(reps),   '副本数应随 QPS 单调增'
assert costs == sorted(costs), '成本应随 QPS 单调增'
assert all(r[3] <= TTFT_SLO_S*1000 + 1e-6 for r in plan), '每一行都必须满足 TTFT SLO'
# 规模经济：QPS 翻 20 倍，单位成本应下降
unit_small = plan[0][4] / plan[0][0]
unit_large = plan[-1][4] / plan[-1][0]
assert unit_large < unit_small, '规模越大，每 QPS 的成本越低（Erlang 规模经济）'
print(f'\n✅ 规模经济：5 QPS 时每 QPS ${unit_small:,.0f}/月，200 QPS 时 ${unit_large:,.0f}/月'
      f'（省 {(1-unit_large/unit_small)*100:.0f}%）')

**🧪 胶囊练习**：实现 `slo_headroom(qps, c, mu, prefill_s, ttft_slo_s)`：
返回在不违反 TTFT SLO 的前提下，**当前配置还能吸收多少倍的流量突增**（返回一个 ≥1 的浮点数；
若当前已违反 SLO 返回 0.0）。用二分或线性扫描均可，精度 0.01 即可。

In [ ]:
def slo_headroom(qps, c, mu, prefill_s, ttft_slo_s):
    # TODO: 找最大的 k，使 mmc_wait(k*qps, mu, c) + prefill_s <= ttft_slo_s
    #       当前已违反则返回 0.0；k 上限取 10.0
    raise NotImplementedError

In [ ]:
# 自测
h = slo_headroom(25, 13, MU_PER_REPLICA, PREFILL_S, TTFT_SLO_S)
assert h >= 1.0, f'按 SLO 规划出的配置至少应有 1.0 倍余量，得到 {h}'
assert mmc_wait(h * 25, MU_PER_REPLICA, 13) + PREFILL_S <= TTFT_SLO_S + 1e-6
# 副本越多，余量越大
h_more = slo_headroom(25, 20, MU_PER_REPLICA, PREFILL_S, TTFT_SLO_S)
assert h_more > h, '更多副本应有更大余量'
# 严重欠配时余量为 0
assert slo_headroom(100, 5, MU_PER_REPLICA, PREFILL_S, TTFT_SLO_S) == 0.0
print(f'25 QPS / 13 副本: 可吸收 {h:.2f}× 突增 | 20 副本: {h_more:.2f}×')
print('✅ 胶囊练习通过：余量是模块 04 自动扩缩「来不来得及」的判据')

In [ ]:
# 📖 胶囊参考答案
def slo_headroom(qps, c, mu, prefill_s, ttft_slo_s):
    if mmc_wait(qps, mu, c) + prefill_s > ttft_slo_s:
        return 0.0
    k, best = 1.0, 1.0
    while k <= 10.0:
        if mmc_wait(k * qps, mu, c) + prefill_s <= ttft_slo_s:
            best = k
        k += 0.01
    return round(best, 2)

---
## 🔧 旁注：真实系统里这些对应什么

- **契约校验** → FastAPI + Pydantic model；vLLM 的 `protocol.py` 定义了完整的 OpenAI schema。
- **健康端点** → K8s `livenessProbe` / `readinessProbe`（模块 03 详述）；vLLM 的 `/health`；自建服务务必自己实现 `/ready`。
- **SSE** → `StreamingResponse(media_type="text/event-stream")`；Ingress 侧 `nginx.ingress.kubernetes.io/proxy-buffering: "off"` + `proxy-read-timeout: "600"`。
- **令牌桶** → Envoy/Kong 的 rate limit filter、Redis + Lua 的分布式令牌桶；OpenAI 的 RPM/TPM 双限就是这个模型。
- **Erlang-C** → 容量规划表；真实数字必须靠压测（`vllm bench serve` / `locust` / `k6`）标定 μ，模型负责外推与形状。
- **准入分级** → Envoy 的 priority + circuit breaker；或网关层按 API key 的 tier 路由。

你在这里算出的「λ→c→成本→TTFT」四元组，就是模块 04 自动扩缩策略的输入，也是模块 05 成本账的基数。

### 小结
- **契约是唯一「一次定型」的决策**；OpenAI 兼容不是抄袭，是协议网络效应。
- **liveness / readiness 语义必须分离**：重启能修的放 liveness，等待或减流能修的放 readiness。LLM 服务的头号事故源就在这。
- **流式必须拆 TTFT / TPOT**；端到端延迟会被输出长度分布主导，不能作为 SLO。SSE 有缓冲、超时、断连三个必踩之坑。
- **四道护栏**：有界队列背压、逐层递减超时、RPM+TPM 双限流、幂等键。LLM 只限 RPM 必被长生成绕过。
- **Erlang-C 给出容量的形状**：排队是利用率的双曲函数、ρ→1 爆炸；并发池越大越省（规模经济）。它是乐观下界，真实数字靠压测标定。

下一站：**模块 03 · Kubernetes 编排** —— 契约定好了，现在让一百个副本在集群里自己活起来。